# D-MTHD Wikipedia benchmark

Right panel: Accelerator **GPU T4 x2**, Internet **On**.

**This is a multi-version run.** Kaggle stops a session at 12 hours, which is less than the full
grid needs. Each version does as much as it can, then stops itself at 11 hours so the packaging
cell below still runs. To continue: add this version's output as an input (Add Input -> Your Work)
and Save & Run All again. The resume path is detected automatically and finished work is skipped.

Download `dmthd_wikipedia_results.tgz` from the output: it holds the metrics, histories, predictions
and generated tables, without the model weights, so it is small.


In [ ]:
import os, subprocess, glob, shutil, zipfile
REPO = "https://github.com/mahdihasanshadi/THESIS.git"
DEST = "/kaggle/working/dmthd-p3"
if not os.path.exists(os.path.join(DEST, "src", "dmthd")):
    r = subprocess.run(["git", "clone", "-q", REPO, DEST])
    if r.returncode != 0:
        shutil.rmtree(DEST, ignore_errors=True)
        tree = glob.glob("/kaggle/input/**/src/dmthd/train_student.py", recursive=True)
        assert tree, "clone failed and no code found among the inputs"
        shutil.copytree(os.path.dirname(os.path.dirname(os.path.dirname(tree[0]))), DEST)
os.chdir(DEST)
subprocess.run(["pip", "install", "-q", "-r", "requirements.txt"])
print("code ready")

In [ ]:
import os, subprocess, glob, sys
env = dict(os.environ, ROOT='/kaggle/working', PYTHONPATH='src', GPU='1', SEEDS='1,2,3',
           COMMITTEES='homo,hetero', MODES='ft,skd,uniform,dmthd', TEACHER_EPOCHS='3', DISAGREEMENT='1')
# resume: find a previous notebook output among the attached inputs (Kaggle mounts them at
# /kaggle/input/notebooks/<user>/<slug>, so the path is detected rather than typed)
env['TIME_BUDGET_S'] = '39600'   # stop launching work after 11 h so the packaging cell below still runs
env['ABLATION_SEEDS'] = '1'      # ablations are supporting evidence: one seed, stated in Limitations
# The implicit specialist is off here. On this corpus it would cost roughly three GPU-hours -
# training it, adapting HateBERT over 68,750 comments at 256 tokens, rebuilding the cache - to
# answer a question the tweet benchmark answers more cheaply and more directly. Wikipedia's
# unique contribution is its annotator agreement, which is what dmthd_dis uses. Set
# SPECIALIST=1 here only after the tweet run shows the specialist is worth the hours.
env['SPECIALIST'] = os.environ.get('SPECIALIST', '0')
cands = sorted({p.replace(chr(92), '/').split('/runs/')[0] for p in glob.glob('/kaggle/input/**/runs/wikipedia', recursive=True)})
env['RESUME_FROM'] = ','.join(cands)   # every attached previous output is merged, richest last
print('resume sources:', cands or '(none: starting fresh)', flush=True)
raw = None   # Wikipedia is downloaded from Figshare by the driver
cmd = ['python', 'kaggle/run_benchmark.py', '--dataset', 'wikipedia', '--stage', 'all'] + (['--raw', raw] if raw else [])
print('running:', ' '.join(cmd), flush=True)
r = subprocess.run(cmd, env=env)
if r.returncode != 0:
    raise SystemExit(f'BENCHMARK FAILED with exit code {r.returncode}: scroll up in this log to the first Traceback')
print('BENCHMARK FINISHED')


In [ ]:
# Package two things: a small results-only archive (metrics, histories, predictions, tables) that is
# easy to download, and the tables themselves. Model weights stay in /kaggle/working for the next
# version to resume from; they are far too large to download.
import glob, os, subprocess, tarfile
ROOT, DS = '/kaggle/working', 'wikipedia'
keep = []
for pat in ('results.json', 'eval_*.json', 'history.csv', 'test_probs.npy', 'test_labels.npy',
            'quantize_eval.json', 'transfer_*.json', 'dropped_teachers.json'):
    keep += glob.glob(f'{ROOT}/runs/{DS}/**/{pat}', recursive=True)
keep += glob.glob(f'{ROOT}/runs/{DS}/*.csv') + glob.glob(f'{ROOT}/cache/{DS}/meta.json') + glob.glob(f'{ROOT}/data/{DS}/report.json')
subprocess.run(['python', '-m', 'dmthd.tables', '--runs', f'{ROOT}/runs/{DS}', '--data', f'{ROOT}/data/{DS}',
                '--out', f'{ROOT}/tables_{DS}'], env=dict(os.environ, PYTHONPATH='src'))
keep += glob.glob(f'{ROOT}/tables_{DS}/*')
out = f'{ROOT}/dmthd_{DS}_results.tgz'
with tarfile.open(out, 'w:gz') as t:
    for f in keep:
        t.add(f, arcname=os.path.relpath(f, ROOT))
print(f'{len(keep)} files -> {out} ({os.path.getsize(out) / 2**20:.1f} MB) : download this one')
print('finished runs:', len(glob.glob(f'{ROOT}/runs/{DS}/*/*/seed*/results.json')))
